# Arm A — Global (Non-Stratified) Baseline.

This notebook builds Arm A from the proposal, the simplest version of the model: every patient, regardless of sex or any other characteristic, is thrown into one pool and a single model is trained on all of them, using both logistic regression and random forest.

The reason to build this plain, ungrouped version first is that later on patients will be split into two groups by sex (Arm B), and separately grouped automatically by an algorithm (Arm C). Whatever results those two grouped approaches produce need something to be measured against, and that something is this ungrouped result. Without it there is no way to say whether grouping the patients actually helped, or whether it was even worth doing.

## 1. init setup

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# confg:
RANDOM_STATE = 42                     
K_OUTER      = 5 
K_INNER      = 5

RANDOM_STATE is fixed at 42 so that all the steps in this code that involve randomness, like shuffling the data or splitting it into training and validation sets, come out the same way every time it is run. That way the experiment can be repeated later and give the exact same result, instead of the numbers shifting around each run.

K_OUTER and K_INNER are both set to 5 because there are only 297 patient records in total, which is not a lot of data. Splitting into 5 groups is a middle ground between having enough patients in each group to be meaningful, and having enough groups that the results are stable and trustworthy. The outer 5 groups are used to score how good the model is; the inner 5 groups are a second split done only within the training data, used just for picking the model's settings, such as what value of C to use for logistic regression. This keeps the data used for the final scoring completely separate from the data used to choose those settings, so the model doesn't get to peek at the answer early.

## 2. Load data define X / y

**leakage

In [3]:
import pandas as pd
df = pd.read_csv("heart+disease/cleveland_clean.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,check
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,False
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,True
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,True
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,False
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,False


num represents the severity of heart disease. and check is representing whether a patient has heart dsease. So we need to remove them, oterwise there will be data leakage

The columns num and check are dropped because check is just num turned into a yes/no answer (check is true whenever num is above 0). If either of these columns were left in as a feature, the model would essentially be shown the answer, or something almost identical to the answer, and wouldn't need to learn anything real about heart disease to get it right. This is called data leakage. A model trained this way would look very accurate, but that accuracy would be fake, and it would be useless on a new patient. So both columns are removed, leaving only the 13 real clinical features to make predictions from.

In [4]:
y = df["check"].astype(int)            # target: 0 = no disease, 1 = disease
X = df.drop(columns=["num", "check"])

print("Samples:", len(df), "  Features:", X.shape[1])
print("Class balance:")
print(y.value_counts().rename({0: "no disease", 1: "disease"}))
print("Positive rate: {:.3f}".format(y.mean()))

Samples: 301   Features: 13
Class balance:
check
no disease    163
disease       138
Name: count, dtype: int64
Positive rate: 0.458


## 3. Feature groups and preprocessing

Feature types are handled per 3.2. Preprocessing lives **inside a Pipeline** so it is re-fit on the
training portion of each fold, -- no statistics leak from validation data.

- **Continuous** (`age, trestbps, chol, thalach, oldpeak`) -> standardised (`StandardScaler`).
- **Nominal** (`cp, restecg, slope, thal`) -> these are category codes, not magnitudes.
- **Binary / count** (`sex, fbs, exang, ca`) -> passed through unchanged (already 0/1, or a 0–3 count).

In [5]:
continuous  = ["age", "trestbps", "chol", "thalach", "oldpeak"]
nominal     = ["cp", "restecg", "slope", "thal"]
passthrough = ["sex", "fbs", "exang", "ca"] # we don't need to do anything

preprocess = ColumnTransformer([
    ("num",  StandardScaler(),                        continuous),
    ("cat",  OneHotEncoder(handle_unknown="ignore"),  nominal),
    ("pass", "passthrough",                           passthrough),
])

The continuous features (age, blood pressure, cholesterol, max heart rate, oldpeak) are standardised because their raw number ranges are very different from each other, for example cholesterol values sit in the hundreds while oldpeak only ranges from 0 to a few. If the numbers are left as they are, a model like logistic regression will end up paying more attention to whichever feature happens to have bigger numbers, not because it's actually more important, just because the numbers are bigger. Standardising puts every feature on the same scale, roughly centred at 0 with similar spread, so the model can compare them fairly.

cp, restecg, slope, and thal are stored as numbers but those numbers actually stand for categories, for example the type of chest pain being 1, 2, 3, or 4. The size of the number doesn't mean anything here, 4 doesn't mean more or worse than 1. So these are one-hot encoded, turning each category into its own separate 0/1 column, so the model doesn't mistakenly assume there's an order or ranking between them.

sex, fbs, and exang are already plain 0/1 yes-or-no values, and ca is a count from 0 to 3 (the number of blocked vessels found). None of these need extra processing, they're passed through unchanged.

All of this scaling and encoding is done inside a Pipeline so that anything learned from the data, like the average or spread used for scaling, is only ever learned from the training portion, never from the validation portion. Otherwise that would also count as data leakage and make the evaluation scores look better than they really are.

## 4. cross-validation folds

The proposal requires **the same fold partitions across all arms** (
    3.5). We assign every patient a
fold id **once**, stratified by the target, and **save it to `fold_id.csv`**. Arms B and C load this
same file, so a patient always sits in the same validation fold regardless of arm. 

In [6]:
skf = StratifiedKFold(n_splits=K_OUTER, shuffle=True, random_state=RANDOM_STATE)

fold_id = np.empty(len(df), dtype=int)
for k, (_, val_idx) in enumerate(skf.split(X, y)):
    fold_id[val_idx] = k

pd.Series(fold_id, name="fold").to_csv("fold_id.csv", index=False)
print("Fold sizes:", np.bincount(fold_id))
print("Positives per fold:", np.bincount(fold_id[y.values == 1]))

Fold sizes: [61 60 60 60 60]
Positives per fold: [28 27 27 28 28]


Each patient's fold assignment is fixed and saved to a file, fold_id.csv, because when Arm B and Arm C are compared against Arm A later, everyone needs to be evaluated on the exact same patients. If each arm split into folds on its own, Arm A might happen to get an easy set of validation patients while Arm B gets a harder set, and then any difference in the results couldn't be trusted, it might just be luck of the draw rather than the grouping method actually working better. Using the same fold file for every arm keeps the comparison fair.

StratifiedKFold is used instead of a plain random split because the number of patients with and without heart disease isn't quite even, 160 versus 137. With a plain random split, one fold could end up with far more healthy patients than sick ones, which would make the accuracy and AUC calculated on that fold unreliable. Stratified splitting keeps the ratio of sick to healthy patients roughly the same in every fold.

## 5. Models and hyperparameter selection

Two classifiers, each with a **small** hyperparameter grid .
The **same procedure is applied in every arm**: hyperparameters are chosen by an **inner
cross-validation** on the training folds only (nested CV), so selection never sees the outer
validation data. 

this code block should be the same across different arms

In [7]:
models = {
    "logreg": (
        Pipeline([("pre", preprocess),
                  ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
        {"clf__C": [0.01, 0.1, 1, 10]},
    ),
    "rf": (
        Pipeline([("pre", preprocess),
                  ("clf", RandomForestClassifier(random_state=RANDOM_STATE))]),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 5, 10]},
    ),
}

Only logistic regression and random forest are used because the point of this study is to compare whether grouping patients helps and how, not to find the single best-performing model. So there's no need for many models, one simple model that's easy to explain (logistic regression) plus one slightly more complex, non-linear model (random forest) is enough for the comparison.

Each model is only given a small number of settings to try (logistic regression tests just 4 values of C, random forest tests just a few combinations of tree count and depth) for the same reason, this experiment isn't about squeezing out the best possible accuracy, it's about keeping the tuning process identical everywhere and looking at how different patient groupings change the outcome. So the tuning is kept deliberately small and simple.

Also, the exact same code and the exact same tuning procedure is used across Arm A, B, and C. This way, if the final results differ between arms, it can be attributed to how the patients were grouped, not to one arm secretly using a better model or more tuning than another.

## 6. Cross-validation

For each model and each outer fold: select hyperparameters by inner CV on the training data, refit on
the full training portion, then predict on the held-out fold. We record accuracy, F1, and ROC-AUC per
fold and store the **out-of-fold predicted probabilities** for every patient.

In [8]:
results_rows = []
oof = {"fold": fold_id, "y": y.values}

for name, (pipe, grid) in models.items():
    accs, f1s, aucs, chosen = [], [], [], []
    oof_proba = np.zeros(len(df))

    for k in range(K_OUTER):
        train, validation = (fold_id != k), (fold_id == k)
        inner = StratifiedKFold(n_splits=K_INNER, shuffle=True, random_state=RANDOM_STATE)
        search = GridSearchCV(pipe, grid, cv=inner, scoring="roc_auc", n_jobs=-1)
        search.fit(X[train], y[train])

        best = search.best_estimator_
        proba = best.predict_proba(X[validation])[:, 1] # take out the ppl that actually has heart disease
        pred  = (proba >= 0.5).astype(int)


        oof_proba[validation] = proba
        accs.append(accuracy_score(y[validation], pred))
        f1s.append(f1_score(y[validation], pred))
        aucs.append(roc_auc_score(y[validation], proba))
        chosen.append(search.best_params_)

    oof[f"proba_{name}"] = oof_proba
    results_rows.append({
        "model": name,
        "accuracy_mean": np.mean(accs), "accuracy_std": np.std(accs),
        "f1_mean":       np.mean(f1s),  "f1_std":       np.std(f1s),
        "rocauc_mean":   np.mean(aucs), "rocauc_std":   np.std(aucs),
    })
    print(f"{name:7s} | ACC {np.mean(accs):.3f} +/- {np.std(accs):.3f}"
          f" | F1 {np.mean(f1s):.3f} +/- {np.std(f1s):.3f}"
          f" | AUC {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}")
    print("         chosen params per fold:", chosen)

logreg  | ACC 0.824 +/- 0.065 | F1 0.788 +/- 0.090 | AUC 0.898 +/- 0.040
         chosen params per fold: [{'clf__C': 0.1}, {'clf__C': 1}, {'clf__C': 0.1}, {'clf__C': 0.1}, {'clf__C': 0.01}]
rf      | ACC 0.820 +/- 0.033 | F1 0.791 +/- 0.057 | AUC 0.907 +/- 0.031
         chosen params per fold: [{'clf__max_depth': 5, 'clf__n_estimators': 200}, {'clf__max_depth': 5, 'clf__n_estimators': 200}, {'clf__max_depth': 5, 'clf__n_estimators': 200}, {'clf__max_depth': 5, 'clf__n_estimators': 200}, {'clf__max_depth': None, 'clf__n_estimators': 200}]


Before scoring each outer fold, an inner 5-fold cross-validation is run inside the training data first to pick the model's settings, rather than just trying different settings directly on the outer validation fold. If the outer validation fold were used to choose the settings, the model would effectively have already seen the data it's about to be scored on, and the score would look better than the model's real ability. The inner CV keeps the entire training and tuning process from ever touching the outer validation fold, so the accuracy, F1, and AUC calculated afterwards can actually be trusted.

A predicted probability above 0.5 is treated as disease present. This is the most standard, straightforward cutoff, and the proposal doesn't call for anything more elaborate, so the default of 0.5 is used here.

Each patient's predicted probability is recorded from the one time they were in the validation fold (an out-of-fold prediction), because these will later need to be compared against Arm B and Arm C's predictions. Only by saving each patient's individual predicted probability is it possible to do a more detailed comparison later, rather than just comparing a single average score.

## 7. Results summary

In [10]:
results = pd.DataFrame(results_rows)
results_display = results.set_index("model").round(3)
results_display

,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
model,,,,,,
logreg,0.824,0.065,0.788,0.090,0.898,0.040
rf,0.820,0.033,0.791,0.057,0.907,0.031


The accuracy, F1, and AUC from each fold are averaged and their standard deviation is also reported, rather than just looking at one fold's result, because a single fold could just be a lucky or unlucky split. Averaging across 5 folds and reporting how much they vary shows both how well the model does on average and how consistent that performance is, which makes the comparison more trustworthy.

## 8. Save outputs to other files

Two files are written for downstream use:

- **`armA_results.csv`** — mean ± std of each metric (goes into your results table).
- **`armA_oof_predictions.csv`** — per-patient fold id, true label, and predicted probability for each
  model. Arms B and C produce the same file on the **same folds**; comparing these three files answers
  RQ1–RQ3.

In [9]:
results.to_csv("armA_results.csv", index=False)
pd.DataFrame(oof).to_csv("armA_predictions.csv", index=False)
print("Saved fold_id.csv, armA_results.csv, armA_predictions.csv")

Saved fold_id.csv, armA_results.csv, armA_predictions.csv


The results and every patient's predicted probability are saved to csv files so that once Arm B and Arm C are run, those files can be pulled up directly to compare against Arm A, without having to rerun Arm A's code again.